<a href="https://colab.research.google.com/github/mobashir-ashraf/Explainable-AI-for-Iris-Liveness-Detection/blob/main/02_Week2_Dataset_Preparation_and_Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 - Dataset Preparation and Baseline Model Development
**Framework:** Explainable AI for Iris Liveness Detection Using Temporal Biometrics and Interpretable Deep Learning

---

## 1. Week 2 Objectives & Scope
This module focuses on transforming raw iris data into optimized tensor matrices and establishing our foundational deep learning classifier:
* **Dataset Preparation:** Designing a robust preprocessing pipeline that standardizes irregular iris image inputs via resizing and min-max normalization.
* **Data Partitioning:** Structuring an un-biased split separating samples into Train (70%), Validation (15%), and Test (15%) subsets to prevent data leakage.
* **Baseline Architecture:** Constructing a ResNet18 convolutional neural network customized for binary classification (Bona Fide/Live vs. Presentation Attack/Spoof).
* **Performance Benchmark:** Evaluating the model using standard classification metrics: Accuracy, Precision, Recall, and F1-score.

---

## 2. Preprocessing & Normalization Methodology
To feed iris textures into a deep neural network, raw image pixels must undergo specific mathematical standardizations:
1. **Spatial Standardizing:** Resizing variable dimensions to a fixed resolution of 224x224 pixels to match the input layer of the ResNet18 backbone.
2. **Tensor Conversion:** Converting raw pixel integer arrays (0 to 255) into floating-point numbers between 0.0 and 1.0.
3. **Channel Statistics Normalization:** Standardizing the dataset using ImageNet channel means [0.485, 0.456, 0.406] and standard deviations [0.229, 0.224, 0.225] to help the model converge faster during training.

In [1]:
# =====================================================================
# TASK: Automated Folder Architecture Setup
# =====================================================================
import os

# Set up the exact folder name you specified
PROJECT_ROOT = "Explainable-AI-for-Iris-Liveness-Detection"
LIVE_DIR = f"{PROJECT_ROOT}/data/raw/live"
SPOOF_DIR = f"{PROJECT_ROOT}/data/raw/spoof"

print("⏳ Python is verifying and building your folder architecture...")

# Automatically create the entire tree structure in one go
os.makedirs(LIVE_DIR, exist_ok=True)
os.makedirs(SPOOF_DIR, exist_ok=True)

print("\n✅ Success! Your folders have been built safely.")
print(f"📁 Created: {LIVE_DIR}")
print(f"📁 Created: {SPOOF_DIR}")
print("\n👉 Next Step: Click the Refresh button (🔄) on your left Files sidebar panel to see them!")

⏳ Python is verifying and building your folder architecture...

✅ Success! Your folders have been built safely.
📁 Created: Explainable-AI-for-Iris-Liveness-Detection/data/raw/live
📁 Created: Explainable-AI-for-Iris-Liveness-Detection/data/raw/spoof

👉 Next Step: Click the Refresh button (🔄) on your left Files sidebar panel to see them!


In [2]:
# =====================================================================
# TASK: Direct Server-to-Server Kaggle Download using New API Tokens
# =====================================================================
import os

# Copied token string assigned directly
os.environ['KAGGLE_API_TOKEN'] = "KGAT_314cf5293980e64c00db1cb2060ef3b3"

# Paths to project structure
LIVE_DIR = "Explainable-AI-for-Iris-Liveness-Detection/data/raw/live"
SPOOF_DIR = "Explainable-AI-for-Iris-Liveness-Detection/data/raw/spoof"

# 1. Download and extract the Real Dataset (CASIA-Iris-Thousand)
print("⏳ Pulling 20,000 Real Images directly from Kaggle into live folder...")
!kaggle datasets download -d sondosaabed/casia-iris-thousand -p {LIVE_DIR} --unzip

# 2. Download and extract the Synthetic Dataset (CASIA-Iris-Syn)
print("\n⏳ Pulling Synthetic Images directly from Kaggle into spoof folder...")
!kaggle datasets download -d monareyhanii/casia-iris-syn -p {SPOOF_DIR} --unzip

print("\n✅ All server-to-server transfers complete! Check your folders!")

⏳ Pulling 20,000 Real Images directly from Kaggle into live folder...
Dataset URL: https://www.kaggle.com/datasets/sondosaabed/casia-iris-thousand
License(s): MIT
100% 491M/491M [00:05<00:00, 97.7MB/s]


⏳ Pulling Synthetic Images directly from Kaggle into spoof folder...
Dataset URL: https://www.kaggle.com/datasets/monareyhanii/casia-iris-syn
License(s): unknown
100% 172M/172M [00:01<00:00, 122MB/s]


✅ All server-to-server transfers complete! Check your folders!


In [3]:
# =====================================================================
# TASK: Generate Physical Presentation Attacks (Printed, Replay, Lens)
# =====================================================================
import os
import cv2
import numpy as np
import glob

# Paths
LIVE_BASE = "Explainable-AI-for-Iris-Liveness-Detection/data/raw/live"
SPOOF_BASE = "Explainable-AI-for-Iris-Liveness-Detection/data/raw/spoof"

# 1. Printed Attack: Simulates paper texture artifacts and lower contrast
def generate_print_attack(img):
    noise = np.random.normal(0, 3, img.shape).astype(np.uint8)
    corrupted = cv2.add(img, noise)
    return cv2.convertScaleAbs(corrupted, alpha=0.85, beta=10)

# 2. Replay Attack: Simulates a screen display with subtle horizontal scanlines
def generate_replay_attack(img):
    h, w, c = img.shape
    moire = np.zeros((h, w, c), dtype=np.uint8)
    for i in range(0, h, 4):
        moire[i:i+2, :, :] = 15
    return cv2.addWeighted(img, 0.9, moire, 0.1, 15)

# 3. Textured Contact Lens Attack: Simulates a fake printed pattern over the iris region
def generate_lens_attack(img):
    h, w, c = img.shape
    mask = np.zeros((h, w), dtype=np.uint8)
    # Draw a simulated textured ring where a contact lens sits over the iris
    cv2.circle(mask, (w//2, h//2), min(w, h)//3, 255, thickness=min(w, h)//6)

    # Generate a high-contrast dot matrix texture pattern for the lens
    lens_texture = np.random.randint(0, 2, (h, w, c), dtype=np.uint8) * 40

    # Apply texture only inside the ring mask area
    textured_img = img.copy()
    textured_img[mask > 0] = cv2.add(textured_img[mask > 0], lens_texture[mask > 0])
    return textured_img

# Grab real source images
real_images = glob.glob(f"{LIVE_BASE}/**/*.jpg", recursive=True) + glob.glob(f"{LIVE_BASE}/**/*.png", recursive=True)
print(f"📸 Found real source images. Generating attack variations...")

# Create strict target directories
os.makedirs(f"{SPOOF_BASE}/physical_print", exist_ok=True)
os.makedirs(f"{SPOOF_BASE}/physical_replay", exist_ok=True)
os.makedirs(f"{SPOOF_BASE}/textured_lens", exist_ok=True)

count = 0
for path in real_images[:1000]:
    img = cv2.imread(path)
    if img is None: continue

    filename = os.path.basename(path)

    # Generate and save all three required attack types
    cv2.imwrite(f"{SPOOF_BASE}/physical_print/print_{filename}", generate_print_attack(img))
    cv2.imwrite(f"{SPOOF_BASE}/physical_replay/replay_{filename}", generate_replay_attack(img))
    cv2.imwrite(f"{SPOOF_BASE}/textured_lens/lens_{filename}", generate_lens_attack(img))

    count += 1

print(f"\n✅ All attacks covered successfully!")
print(f"📁 Created {count} Print attacks, {count} Replay attacks, and {count} Textured Lens attacks inside your 'spoof' directory!")

📸 Found real source images. Generating attack variations...

✅ All attacks covered successfully!
📁 Created 1000 Print attacks, 1000 Replay attacks, and 1000 Textured Lens attacks inside your 'spoof' directory!


In [4]:
# =====================================================================
# TASK: Week 2 Dataset Preprocessing and Train/Val/Test Splitting
# =====================================================================
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# 1. Configuration Settings
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224) # Standard input spatial resolution for ResNet18

# 2. Define the exact Preprocessing Pipeline (Resize + Normalize)
preprocess_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Custom PyTorch Dataset Loader
class IrisLivenessDataset(Dataset):
    def __init__(self, live_dir, spoof_dir, transform=None):
        self.transform = transform
        self.file_list = []
        self.labels = []

        # Gather all genuine samples (Class 0: Live)
        live_images = (glob.glob(os.path.join(live_dir, "**/*.jpg"), recursive=True) +
                       glob.glob(os.path.join(live_dir, "**/*.png"), recursive=True))
        for img_path in live_images:
            self.file_list.append(img_path)
            self.labels.append(0) # 0 = Live

        # Gather all attack samples (Class 1: Spoof)
        spoof_images = (glob.glob(os.path.join(spoof_dir, "**/*.jpg"), recursive=True) +
                        glob.glob(os.path.join(spoof_dir, "**/*.png"), recursive=True))
        for img_path in spoof_images:
            self.file_list.append(img_path)
            self.labels.append(1) # 1 = Spoof

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            image = Image.new('RGB', IMAGE_SIZE, color=0)

        if self.transform:
            image = self.transform(image)

        return image, label

# 4. Initialize Data Sources
LIVE_PATH = "Explainable-AI-for-Iris-Liveness-Detection/data/raw/live"
SPOOF_PATH = "Explainable-AI-for-Iris-Liveness-Detection/data/raw/spoof"

full_dataset = IrisLivenessDataset(LIVE_PATH, SPOOF_PATH, transform=preprocess_transforms)
total_samples = len(full_dataset)

if total_samples == 0:
    print("⚠️ Error: No images found. Check your file paths or refresh the panel.")
else:
    # Calculate strict target splits: 70% Train, 15% Validation, 15% Test
    train_size = int(0.70 * total_samples)
    val_size = int(0.15 * total_samples)
    test_size = total_samples - train_size - val_size

    # Set seed for reproducible splitting
    generator = torch.Generator().manual_seed(42)
    train_set, val_set, test_set = random_split(full_dataset, [train_size, val_size, test_size], generator=generator)

    # 5. Create the DataLoaders for PyTorch Model Streams
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print("📊 --- Dataset Splits Prepared Successfully ---")
    print(f"📦 Total Indexed Dataset Size: {total_samples} images")
    print(f"🏋️ Training Set Size:         {len(train_set)} images (70%)")
    print(f"🧪 Validation Set Size:       {len(val_set)} images (15%)")
    print(f"🔬 Testing Set Size:          {len(test_set)} images (15%)")

📊 --- Dataset Splits Prepared Successfully ---
📦 Total Indexed Dataset Size: 33000 images
🏋️ Training Set Size:         23100 images (70%)
🧪 Validation Set Size:       4950 images (15%)
🔬 Testing Set Size:          4950 images (15%)


In [5]:
# =====================================================================
# TASK: Week 2 Baseline Model Definition & Training Loop
# =====================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 1. Force PyTorch to use active T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using execution device: {device}")

# 2. Initialize ResNet18 Architecture with standard ImageNet weights
print("🧠 Loading pre-trained ResNet18 backbone...")
baseline_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modify the final Fully Connected layer for binary classification
# ResNet18 outputs 512 features, we map it to 2 classes (0: Live, 1: Spoof)
num_features = baseline_model.fc.in_features
baseline_model.fc = nn.Linear(num_features, 2)

# Transfer model onto the T4 GPU
baseline_model = baseline_model.to(device)

# 3. Define Standard Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=0.0001)

# 4. Run a Baseline Training Loop (Let's run 2 Epochs to establish baseline performance)
NUM_EPOCHS = 2

print("\n🏋️ Starting Baseline Model Training Pipeline...")
for epoch in range(NUM_EPOCHS):
    baseline_model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Zero out the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = baseline_model(images)
        loss = criterion(outputs, labels)

        # Backward pass + Optimize weights
        loss.backward()
        optimizer.step()

        # Calculate statistics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_predictions += torch.sum(preds == labels.data)
        total_samples += labels.size(0)

        # Print progress update every 100 batches
        if (batch_idx + 1) % 100 == 0:
            current_acc = (correct_predictions.double() / total_samples) * 100
            print(f"   [Epoch {epoch+1}/{NUM_EPOCHS} | Batch {batch_idx+1}/{len(train_loader)}] "
                  f"Batch Loss: {loss.item():.4f} | Running Acc: {current_acc:.2f}%")

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct_predictions.double() / len(train_loader.dataset)) * 100
    print(f"🏁 Epoch {epoch+1} Complete -> Average Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%\n")

print("✅ Baseline ResNet18 model has been trained successfully!")

🚀 Using execution device: cuda
🧠 Loading pre-trained ResNet18 backbone...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 207MB/s]



🏋️ Starting Baseline Model Training Pipeline...
   [Epoch 1/2 | Batch 100/722] Batch Loss: 0.0205 | Running Acc: 96.22%
   [Epoch 1/2 | Batch 200/722] Batch Loss: 0.0207 | Running Acc: 97.28%
   [Epoch 1/2 | Batch 300/722] Batch Loss: 0.0102 | Running Acc: 98.01%
   [Epoch 1/2 | Batch 400/722] Batch Loss: 0.1862 | Running Acc: 98.38%
   [Epoch 1/2 | Batch 500/722] Batch Loss: 0.0006 | Running Acc: 98.55%
   [Epoch 1/2 | Batch 600/722] Batch Loss: 0.0013 | Running Acc: 98.73%
   [Epoch 1/2 | Batch 700/722] Batch Loss: 0.0006 | Running Acc: 98.86%
🏁 Epoch 1 Complete -> Average Loss: 0.0318 | Accuracy: 98.90%

   [Epoch 2/2 | Batch 100/722] Batch Loss: 0.0002 | Running Acc: 99.88%
   [Epoch 2/2 | Batch 200/722] Batch Loss: 0.0000 | Running Acc: 99.94%
   [Epoch 2/2 | Batch 300/722] Batch Loss: 0.0003 | Running Acc: 99.92%
   [Epoch 2/2 | Batch 400/722] Batch Loss: 0.0003 | Running Acc: 99.89%
   [Epoch 2/2 | Batch 500/722] Batch Loss: 0.0000 | Running Acc: 99.88%
   [Epoch 2/2 | Batch 60

In [6]:
# =====================================================================
# TASK: Week 2 Performance Evaluation Metrics Pipeline
# =====================================================================
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Switch model to evaluation mode
baseline_model.eval()

all_preds = []
all_labels = []

print("🧪 Running evaluation pass on the Test dataset split...")

# 2. Iterate through test batches without tracking gradients (saves memory)
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        # Forward pass through the trained ResNet18 network
        outputs = baseline_model(images)
        _, preds = torch.max(outputs, 1)

        # Collect values on CPU for evaluation
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Convert arrays to numpy
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 3. Calculate the strict target metrics requested in the PDF
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='binary')
recall = recall_score(all_labels, all_preds, average='binary')
f1 = f1_score(all_labels, all_preds, average='binary')

# 4. Display the Official Performance Report
print("\n📊 ==================================================")
print("🔬       BASELINE EVALUATION METRICS         ")
print("====================================================")
print(f"✅ Test Accuracy:  {accuracy * 100:.2f}%")
print(f"🎯 Test Precision: {precision * 100:.2f}%  (Ability to avoid false alarms)")
print(f"📢 Test Recall:    {recall * 100:.2f}%  (Ability to catch all spoofs)")
print(f"⚖️  Test F1-Score:  {f1 * 100:.2f}%  (Harmonic balance of both)")
print("====================================================\n")

print("📋 Detailed Class-by-Class Breakdown:")
print(classification_report(all_labels, all_preds, target_names=['0: Live', '1: Spoof']))

🧪 Running evaluation pass on the Test dataset split...

📊 ==================================================
🔬       BASELINE EVALUATION METRICS         
✅ Test Accuracy:  99.92%
🎯 Test Precision: 99.84%  (Ability to avoid false alarms)
📢 Test Recall:    99.95%  (Ability to catch all spoofs)
⚖️  Test F1-Score:  99.90%  (Harmonic balance of both)

📋 Detailed Class-by-Class Breakdown:
              precision    recall  f1-score   support

     0: Live       1.00      1.00      1.00      3032
    1: Spoof       1.00      1.00      1.00      1918

    accuracy                           1.00      4950
   macro avg       1.00      1.00      1.00      4950
weighted avg       1.00      1.00      1.00      4950

